# 📘 Python 异步编程教程

一份全面、严谨、自包含的 Python 异步编程教程。

**你将学到：**
- 第一部分：异步编程简介
- 第二部分：事件循环
- 第三部分：`async def` 和 `await`
- 第四部分：在 Cursor 中运行异步代码
- 第五部分：Gradio 回调中的异步代码
- 第六部分：进阶 – `asyncio.gather`
- 第七部分：技巧、陷阱与实用方法
- 第八部分：练习题

---
## 第一部分：异步编程简介 – 是什么、为什么、怎么用？

### 🔄 什么是 Python 异步编程？

异步 Python 是一种编写**不会阻塞**的代码的方式。当等待某个操作完成时（比如网络请求、文件读取或 sleep），异步代码不会停止一切，而是让其他任务在等待期间继续运行。

这对于 **I/O 密集型**操作特别有用——这类操作之所以慢，是因为外部资源（如网络或磁盘）的延迟，而不是因为 CPU 负载。

### 🤹 异步 vs 多线程 vs 多进程

| 特性 | `asyncio`（异步） | 多线程（Threads） | 多进程（Multiprocessing） |
|------|------------------|------------------|------------------------|
| 适用场景 | I/O 密集型任务 | I/O 密集型（有时） | CPU 密集型任务 |
| 并发模型 | 协作式 | 抢占式 | 真正的并行 |
| 开销 | 低 | 中等 | 高 |
| 复杂度 | 中等 | 中等 | 高 |
| 受 GIL 影响 | 是 | 是 | 否 |

- **异步**是单线程的，但可以处理数千个并发 I/O 任务。
- **多线程**允许同时执行多个操作，但可能存在竞态条件，且受限于 GIL（全局解释器锁）。
- **多进程**绕过了 GIL，在多个进程中运行，适合 CPU 密集型任务。

---
## 第二部分：🧠 事件循环

把事件循环想象成一个**指挥家**，管理着多个乐器（任务）。它检查哪个任务准备好了，然后运行它；当任务需要等待时就暂停它，切换到另一个任务。

**幕后原理：**
- `say_hello()` 是一个**协程（coroutine）**。
- `asyncio.run()` 启动一个**事件循环**。
- 当执行到 `await asyncio.sleep(1)` 时，事件循环会**暂停** `say_hello`，并可以运行其他协程。

In [1]:
import asyncio

async def say_hello():
    print("你好")
    await asyncio.sleep(1)  # 在这里暂停，切换到其他任务
    print("世界")

# 在 notebook 中，可以直接使用顶层 await！
await say_hello()

你好
世界


---
## 第三部分：✅ `async def` 和 `await`

### 基本用法

**规则：**
- ✅ 使用 `async def` 来定义协程
- ✅ `await` 只能在 `async def` 内部使用
- ❌ 不能在脚本或模块的顶层使用 `await` —— 需要用 `asyncio.run()` 包裹
- ✅ 可以用 `return` 从异步函数中返回值
- ✅ 在 notebook 中支持顶层 `await`！

In [2]:
import asyncio

async def greet(name):
    print(f"你好，{name}")
    await asyncio.sleep(1)
    print(f"再见，{name}")

async def main():
    await greet("小明")

# 在 notebook 中：
await main()

你好，小明
再见，小明


### 💡 忘记写 `await` 会怎样？

运行下面的代码观察一下——你会得到一个协程对象，而不是执行函数！

In [3]:
# 常见错误：忘记 await
result = greet("小红")  # 没有 await！
print(f"没有 await 的结果: {result}")
print(f"类型: {type(result)}")

# 正确的方式：
print("\n使用 await：")
await greet("小红")

没有 await 的结果: <coroutine object greet at 0x10cbcf1d0>
类型: <class 'coroutine'>

使用 await：
你好，小红
再见，小红


---
## 第四部分：在 Cursor 中运行异步代码

### ✅ 从 Python 模块运行（例如 `main.py`）

```python
# main.py
import asyncio

async def work():
    await asyncio.sleep(1)
    print("异步完成！")

if __name__ == "__main__":
    asyncio.run(work())
```

✅ 在终端运行 `python main.py` 即可。

### ✅ 从 Notebook 运行（在 Cursor 或 Jupyter 中）

Notebook 支持顶层 `await`！**不需要** `asyncio.run()`。

💡 如果你在嵌入事件循环的场景中（如某些服务器或 LLM 应用），可以使用 `nest_asyncio`。

In [4]:
# 在 notebook 中可以直接这样写——不需要 asyncio.run()！
import asyncio

async def hello():
    await asyncio.sleep(1)
    print("来自 notebook 的问候！")

await hello()

来自 notebook 的问候！


---
## 第五部分：Gradio 回调中的异步代码

Gradio 允许你直接使用 `async def` 作为事件处理函数（如按钮点击回调）。

🎉 它能直接工作！Gradio 底层使用了 `asyncio`。

> **注意：** 确保已安装 gradio：`uv pip install gradio`

In [6]:
# 如果已安装 gradio，取消注释并运行
import gradio as gr
import asyncio

async def slow_response(name):
    await asyncio.sleep(2)
    return f"你好，{name}！（等待之后）"

gr.Interface(fn=slow_response, inputs="text", outputs="text").launch()

/Users/ai-hub/coding/g-1pv/0_229_Python_Hard_Part/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


---
## 第六部分：进阶异步 – `asyncio.gather`

### 并发运行多个协程

`asyncio.gather()` 让所有任务*并行*运行（就 I/O 等待而言）。这是异步编程真正发光的地方！

In [7]:
import asyncio
import time

async def task(name, delay):
    print(f"⏳ {name} 开始执行（需要 {delay} 秒）")
    await asyncio.sleep(delay)
    print(f"✅ {name} 完成，耗时 {delay} 秒")
    return name

async def main():
    start = time.time()
    results = await asyncio.gather(
        task("A", 1),
        task("B", 2),
        task("C", 3)
    )
    elapsed = time.time() - start
    print(f"\n所有结果: {results}")
    print(f"总耗时: {elapsed:.1f} 秒（不是 6 秒！）")

await main()

⏳ A 开始执行（需要 1 秒）
⏳ B 开始执行（需要 2 秒）
⏳ C 开始执行（需要 3 秒）
✅ A 完成，耗时 1 秒
✅ B 完成，耗时 2 秒
✅ C 完成，耗时 3 秒

所有结果: ['A', 'B', 'C']
总耗时: 3.0 秒（不是 6 秒！）


### 🔍 对比：顺序执行 vs 并发执行

让我们看看逐个运行任务和使用 `gather` 的区别：

In [8]:
import asyncio
import time

async def slow_task(name, delay):
    await asyncio.sleep(delay)
    return f"{name} 完成"

# 顺序执行
start = time.time()
r1 = await slow_task("A", 1)
r2 = await slow_task("B", 1)
r3 = await slow_task("C", 1)
print(f"顺序执行: {time.time() - start:.1f} 秒 → {[r1, r2, r3]}")

# 并发执行
start = time.time()
results = await asyncio.gather(
    slow_task("A", 1),
    slow_task("B", 1),
    slow_task("C", 1)
)
print(f"并发执行: {time.time() - start:.1f} 秒 → {results}")

顺序执行: 3.0 秒 → ['A 完成', 'B 完成', 'C 完成']
并发执行: 1.0 秒 → ['A 完成', 'B 完成', 'C 完成']


---
## 第七部分：技巧、陷阱与实用方法

### ✅ 实用技巧

- 使用 `asyncio.gather` 并行化 I/O 密集型调用。
- 使用 `async for` 和 `async with` 处理异步迭代器和上下文管理器。
- 如果需要更高级的异步框架，可以使用 `anyio` 或 `trio`。

### ⚠️ 常见陷阱

| 陷阱 | 解决方法 |
|------|--------|
| 在脚本顶层使用 `await` | 使用 `asyncio.run()` |
| 混合使用 `threading` 和 `asyncio` | 尽量避免，或谨慎使用 |
| 忘记写 `await` | 你会得到一个协程对象，什么也不会执行 |
| 使用阻塞调用（如 `time.sleep`） | 改用 `await asyncio.sleep()` |

### 🔧 调试技巧

使用以下代码验证是否在运行中的事件循环内：

In [ ]:
import asyncio

# 检查事件循环是否正在运行
loop = asyncio.get_running_loop()
print(f"事件循环正在运行: {loop.is_running()}")
print(f"循环类型: {type(loop).__name__}")

### 🚫 陷阱演示：阻塞式 vs 非阻塞式 sleep

In [ ]:
import asyncio
import time

# 错误示范：这会阻塞整个事件循环！
async def bad_example():
    print("开始错误示范（使用 time.sleep）...")
    time.sleep(2)  # ❌ 阻塞一切！
    print("错误示范结束")

# 正确示范：这让其他任务在等待期间继续运行
async def good_example():
    print("开始正确示范（使用 asyncio.sleep）...")
    await asyncio.sleep(2)  # ✅ 非阻塞
    print("正确示范结束")

print("=== 错误（阻塞式） ===")
await bad_example()
print("\n=== 正确（非阻塞式） ===")
await good_example()

---
## 第八部分：🏋️ 练习题

在下面的代码单元中完成这些练习！每个练习都有描述和供你编写解答的代码单元。

---

### 练习 1 – 基本协程

编写一个协程：
- 接收一个名字参数
- 等待 2 秒
- 打印 `"你好，[名字]！"`

In [ ]:
import asyncio

# 练习 1：在这里编写你的协程
async def greet(name):
    # TODO: 实现这个函数
    pass

# 测试：
# await greet("小明")

<details>
<summary>💡 点击查看答案</summary>

```python
async def greet(name):
    await asyncio.sleep(2)
    print(f"你好，{name}！")

await greet("小明")
```
</details>

---
### 练习 2 – 并行协程

编写三个异步函数：
- `fetch_data()` – 等待 1 秒，返回 `"数据已获取"`
- `process_data()` – 等待 1 秒，返回 `"数据已处理"`
- `save_data()` – 等待 1 秒，返回 `"数据已保存"`

使用 `asyncio.gather()` 并发运行这三个函数。

In [ ]:
import asyncio

# 练习 2：在这里编写你的函数
async def fetch_data():
    # TODO
    pass

async def process_data():
    # TODO
    pass

async def save_data():
    # TODO
    pass

# 使用 asyncio.gather 运行这三个函数
# results = await asyncio.gather(fetch_data(), process_data(), save_data())
# print(results)

<details>
<summary>💡 点击查看答案</summary>

```python
async def fetch_data():
    await asyncio.sleep(1)
    return "数据已获取"

async def process_data():
    await asyncio.sleep(1)
    return "数据已处理"

async def save_data():
    await asyncio.sleep(1)
    return "数据已保存"

results = await asyncio.gather(fetch_data(), process_data(), save_data())
print(results)  # 总耗时约 1 秒，而不是 3 秒！
```
</details>

---
### 练习 3 – 异步倒计时

编写一个协程，从 5 倒数到 1，每个数字之间等待 1 秒。

In [ ]:
import asyncio

# 练习 3：异步倒计时
async def countdown(n):
    # TODO: 实现从 n 到 1 的倒计时
    pass

# 测试：
# await countdown(5)

<details>
<summary>💡 点击查看答案</summary>

```python
async def countdown(n):
    for i in range(n, 0, -1):
        print(i)
        await asyncio.sleep(1)
    print("出发！🚀")

await countdown(5)
```
</details>

---
### 练习 4 – 对比阻塞式 vs 异步

运行下面的代码，比较两种方式各需要多长时间。
分别对阻塞式和异步版本循环 3 次，并计时比较。

In [ ]:
import asyncio
import time

def blocking():
    time.sleep(1)
    print("阻塞式完成")

async def non_blocking():
    await asyncio.sleep(1)
    print("异步完成")

# 练习 4：分别计时 3 次迭代
# 阻塞式版本：
# start = time.time()
# for _ in range(3):
#     blocking()
# print(f"阻塞式总耗时: {time.time() - start:.1f} 秒")

# 异步版本（使用 asyncio.gather！）：
# start = time.time()
# await asyncio.gather(*[non_blocking() for _ in range(3)])
# print(f"异步总耗时: {time.time() - start:.1f} 秒")

<details>
<summary>💡 点击查看答案</summary>

```python
# 阻塞式版本：约 3 秒
start = time.time()
for _ in range(3):
    blocking()
print(f"阻塞式总耗时: {time.time() - start:.1f} 秒")

# 异步版本：约 1 秒
start = time.time()
await asyncio.gather(*[non_blocking() for _ in range(3)])
print(f"异步总耗时: {time.time() - start:.1f} 秒")
```

阻塞式版本耗时约 3 秒（顺序执行），而异步版本仅需约 1 秒（并发执行）！
</details>

---
### 练习 5 – 构建 Gradio 异步应用

创建一个 Gradio 应用：
- 接收一个名字
- 等待 2 秒
- 返回一个问候语

尝试在同步和异步处理函数之间切换，测量响应速度的差异。

> 先安装 gradio：`uv pip install gradio`

In [ ]:
# 练习 5：Gradio 异步应用
# import gradio as gr
# import asyncio

# 同步版本
# def sync_greet(name):
#     import time
#     time.sleep(2)
#     return f"你好，{name}！"

# 异步版本
# async def async_greet(name):
#     await asyncio.sleep(2)
#     return f"你好，{name}！"

# 两个版本都试试：
# gr.Interface(fn=async_greet, inputs="text", outputs="text").launch()

---
## 🎓 总结

| 概念 | 核心要点 |
|------|--------|
| `async def` | 定义协程 |
| `await` | 暂停协程，让其他任务运行 |
| `asyncio.run()` | 启动事件循环（在脚本中使用） |
| 顶层 `await` | 在 notebook 中可直接使用 |
| `asyncio.gather()` | 并发运行多个协程 |
| `asyncio.sleep()` | 非阻塞式休眠 |
| `time.sleep()` | ❌ 阻塞式——在异步代码中应避免使用 |

**祝你异步编程愉快！🚀**